# Phase 2 agentic (LangGraph) vs Phase 1 — Amazon Reviews 2023 demo

**Phase 1:** one forward pass — **LLaMA-3-8B + your LoRA**, review **text only** (same prompts as `llama_sentiment_baseline_train.ipynb`).

**Phase 2 (implemented demo):** a small **multi-step graph** (LangGraph) with the same backbone model:
1. **Analyst** — text-only review analysis (hypothesis **H** + draft rating).
2. **RAG Prover** — *toy grounding*: product **title + details** from metadata (stand-in for LanceDB retrieval).
3. **Visual Verifier** — **BLIP** image caption as **Ev** (stand-in for a vision auditor).
4. **Critic** — second LLM call that reconciles H with **Gf** + **Ev** and outputs the **final** `Sentiment (1-5)`.

This is a **faithful skeleton** of the architecture in `Agentic_Sentiment_LLaMA3.html` §§12–13, not production LanceDB/LLaVA.

**Default eval size:** ~**1000** reviews with **metadata join + product image URL** (BLIP caption for Phase 2). That run needs **many GPU hours** (Phase 1 = 1× LLM/review; Phase 2 graph = Analyst + Critic = 2× LLM/review). Lower **`N_SAMPLES`** in config for a quick smoke test.

**Prerequisite:** trained adapter from `llama_sentiment_baseline_train.ipynb` (`output_final.zip` or `./output/final`).

### Google Colab

1. **Runtime → Change runtime type → GPU** (T4 / L4 recommended; CPU will be very slow and may OOM on LLaMA).
2. Upload **`output_final.zip`** to **`/content/`** (Files pane), or unzip so **`/content/output/final`** exists.
3. **Llama-3** is gated: create a [HF token](https://huggingface.co/settings/tokens), accept the model card, then either:
   - Colab **Secrets** (key `HF_TOKEN`) — the next cell logs in automatically, or
   - Run `huggingface-cli login` in a terminal cell.
4. If **`bitsandbytes`** errors after install, use **Runtime → Restart session**, then **run all** from the top.


## 1. Install


In [ ]:
%pip install -q datasets pyarrow transformers accelerate peft safetensors sentencepiece huggingface_hub
%pip install -q "bitsandbytes>=0.43.0"
%pip install -q pandas pillow requests matplotlib timm tqdm
%pip install -q langgraph


## 1b. Colab: GPU check + Hugging Face login (Llama-3)


In [ ]:
import os
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from huggingface_hub import login

_tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not _tok:
    try:
        from google.colab import userdata

        _tok = userdata.get("HF_TOKEN")
    except Exception:
        _tok = None
if _tok:
    login(token=_tok, add_to_git_credential=False)
    print("HF: logged in from token.")
else:
    print(
        "HF: no token found. Add Colab Secret HF_TOKEN or set HF_TOKEN env; else run: !huggingface-cli login"
    )


## 2. Configuration


In [ ]:
import os
import random

try:
    import google.colab  # noqa: F401

    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

# Colab: cwd is usually /content — put zip here or use Drive copy
if _IN_COLAB:
    _root = "/content"
    ADAPTER_ZIP = os.path.join(_root, "output_final.zip")
    ADAPTER_PATH = os.environ.get("ADAPTER_PATH", os.path.join(_root, "output", "final"))
else:
    ADAPTER_ZIP = os.environ.get("ADAPTER_ZIP", "output_final.zip")
    ADAPTER_PATH = os.environ.get("ADAPTER_PATH", "./output/final")

BASE_MODEL = "meta-llama/Meta-Llama-3-8B"

REVIEW_CONFIG = "raw_review_All_Beauty"
META_CONFIG = "raw_meta_All_Beauty"
N_SAMPLES = 1000
SEED = 42
# Keep only rows with metadata, a non-trivial review, and at least one product image URL (Phase 2 uses BLIP).
REQUIRE_IMAGE_URL = True
REQUIRE_META_FIELDS = True
MIN_REVIEW_CHARS = 20
# Checkpoint CSV during the long LLM loop (resume: load df partial and skip completed parent_asin).
RESULTS_CSV = os.environ.get("RESULTS_CSV", "phase1_vs_phase2_amazon2023_results.csv")
CHECKPOINT_EVERY = 25
# Cap metadata indexing pass (raise if you need every SKU in huge categories).
META_INDEX_MAX_ROWS = int(os.environ.get("META_INDEX_MAX_ROWS", "500000"))

MAX_NEW_TOKENS_ANALYST = 128
MAX_NEW_TOKENS_CRITIC = 192
POOL_SIZE = 5
PROMPT_MAX_LENGTH = 2048

BLIP_MODEL = "Salesforce/blip-image-captioning-base"
MAX_IMAGE_BYTES = 2_000_000

random.seed(SEED)
print("Colab:", _IN_COLAB, "| ADAPTER_ZIP:", ADAPTER_ZIP, "| ADAPTER_PATH:", ADAPTER_PATH)


## 3. Load Amazon Reviews 2023 (reviews + metadata + image URLs)

Hugging Face `datasets` 3.x **no longer runs dataset Python scripts**. This notebook loads **reviews** from **`json`** (`raw/review_categories/{Category}.jsonl`) and **metadata** from **`parquet`** under `raw_meta_{Category}/` (no `trust_remote_code`).


In [ ]:
import re
import warnings
from typing import Any, Dict, List, Optional

import pandas as pd
import requests
from datasets import load_dataset
from huggingface_hub import list_repo_files
from PIL import Image
from io import BytesIO

warnings.filterwarnings("ignore")

AMAZON2023_REPO = "McAuley-Lab/Amazon-Reviews-2023"
AMAZON2023_MAIN = f"https://huggingface.co/datasets/{AMAZON2023_REPO}/resolve/main"


def amazon2023_review_jsonl_url(review_config: str) -> str:
    if not review_config.startswith("raw_review_"):
        raise ValueError(f"Expected raw_review_* config, got {review_config!r}")
    cat = review_config.replace("raw_review_", "", 1)
    return f"{AMAZON2023_MAIN}/raw/review_categories/{cat}.jsonl"


def load_amazon2023_meta_parquet(meta_config: str):
    prefix = f"{meta_config}/"
    paths = sorted(
        p
        for p in list_repo_files(AMAZON2023_REPO, repo_type="dataset")
        if p.startswith(prefix) and p.endswith(".parquet")
    )
    if not paths:
        raise FileNotFoundError(
            f"No .parquet under {prefix!r} in {AMAZON2023_REPO}. Pick a raw_meta_* category present on the Hub."
        )
    return load_dataset(
        "parquet",
        data_files=[f"{AMAZON2023_MAIN}/{p}" for p in paths],
        split="train",
    )


from tqdm.auto import tqdm

print("Loading reviews:", REVIEW_CONFIG)
rev_split = load_dataset(
    "json", data_files=amazon2023_review_jsonl_url(REVIEW_CONFIG), split="train"
)
print("Loading metadata:", META_CONFIG)
meta_ds = load_amazon2023_meta_parquet(META_CONFIG)

meta_by_asin: Dict[str, Any] = {}
n_meta = len(meta_ds)
scan_upto = min(n_meta, META_INDEX_MAX_ROWS)
if scan_upto < n_meta:
    print(
        f"Warning: indexing first {scan_upto} of {n_meta} metadata rows; set META_INDEX_MAX_ROWS to scan more."
    )
print("Indexing metadata by ASIN (one pass)...")
for i in tqdm(range(scan_upto), desc="meta"):
    r = meta_ds[i]
    pa = str(r.get("parent_asin") or r.get("asin") or "")
    if pa:
        meta_by_asin[pa] = r
print("Unique ASINs in indexed metadata:", len(meta_by_asin))

def first_image_url(meta: dict) -> Optional[str]:
    im = meta.get("images") or {}
    if isinstance(im, dict):
        for k in ("hi_res", "large", "thumb"):
            for u in im.get(k) or []:
                if u and isinstance(u, str) and u.startswith("http"):
                    return u
    return None

def safe_details(meta: dict) -> str:
    d = meta.get("details")
    if d is None:
        return ""
    if isinstance(d, str):
        return d[:800]
    return str(d)[:800]


def row_passes_filters(rev_row: dict, meta: dict) -> bool:
    text = (rev_row.get("text") or "").strip()
    if len(text) < MIN_REVIEW_CHARS:
        return False
    if not meta:
        return False
    if REQUIRE_META_FIELDS:
        has_title = bool((meta.get("title") or rev_row.get("title") or "").strip())
        has_details = bool(safe_details(meta).strip())
        if not (has_title or has_details):
            return False
    if REQUIRE_IMAGE_URL:
        u = first_image_url(meta)
        if not (isinstance(u, str) and u.startswith("http")):
            return False
    return True


def review_to_record(row: dict, meta: dict) -> Dict[str, Any]:
    pa = str(row.get("parent_asin") or row.get("asin") or "")
    title = (row.get("title") or meta.get("title") or "").strip()
    text = (row.get("text") or "").strip()
    rating = row.get("rating")
    try:
        gt = int(round(float(rating)))
    except (TypeError, ValueError):
        gt = 3
    gt = max(1, min(5, gt))
    cat = meta.get("main_category") or REVIEW_CONFIG.replace("raw_review_", "").replace("_", " ") or "Unknown"
    return {
        "parent_asin": pa,
        "review_text": text[:4000],
        "ground_truth_stars": gt,
        "main_category": cat,
        "meta_title": (meta.get("title") or title)[:500],
        "details": safe_details(meta),
        "image_url": first_image_url(meta),
    }


target = min(N_SAMPLES, len(rev_split))
rng = random.Random(SEED)
order = list(range(len(rev_split)))
rng.shuffle(order)
records: List[Dict[str, Any]] = []
seen_keys = set()
for j in tqdm(order, desc="pick reviews"):
    if len(records) >= target:
        break
    rev_row = rev_split[int(j)]
    pa = str(rev_row.get("parent_asin") or rev_row.get("asin") or "")
    if not pa:
        continue
    meta = meta_by_asin.get(pa)
    if meta is None:
        continue
    if not row_passes_filters(rev_row, meta):
        continue
    txt = (rev_row.get("text") or "")[:400]
    key = (pa, txt)
    if key in seen_keys:
        continue
    seen_keys.add(key)
    records.append(review_to_record(rev_row, meta))

if len(records) < target:
    print(
        f"Warning: only {len(records)} rows matched filters (wanted {target}). "
        "Try REQUIRE_IMAGE_URL=False, another category, or larger META_INDEX_MAX_ROWS."
    )

df_raw = pd.DataFrame(records).reset_index(drop=True)
print(df_raw[["ground_truth_stars", "main_category"]].head())
print("Rows:", len(df_raw), "| with image URL:", df_raw["image_url"].str.startswith("http", na=False).sum())


## 4. BLIP captions (Visual Verifier input **Ev**)

Frees GPU memory before loading LLaMA.


In [ ]:
from tqdm.auto import tqdm

caption_by_idx: Dict[int, str] = {}
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration

device_blip = "cuda" if torch.cuda.is_available() else "cpu"
print("BLIP device:", device_blip)
proc = BlipProcessor.from_pretrained(BLIP_MODEL)
blip = BlipForConditionalGeneration.from_pretrained(BLIP_MODEL).to(device_blip)
blip.eval()

def caption_from_url(url: str) -> str:
    try:
        r = requests.get(url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        if len(r.content) > MAX_IMAGE_BYTES:
            return ""
        img = Image.open(BytesIO(r.content)).convert("RGB")
        inputs = proc(images=img, return_tensors="pt").to(device_blip)
        out = blip.generate(**inputs, max_new_tokens=40)
        return proc.tokenizer.decode(out[0], skip_special_tokens=True).strip()
    except Exception as e:
        return f"[image unavailable: {e}]"

for pos in tqdm(range(len(df_raw)), desc="BLIP captions"):
    row = df_raw.iloc[pos]
    u = row.get("image_url")
    caption_by_idx[pos] = caption_from_url(u) if isinstance(u, str) and u.startswith("http") else ""

del blip, proc
if torch.cuda.is_available():
    torch.cuda.empty_cache()

df_raw["image_caption"] = [caption_by_idx.get(i, "") for i in range(len(df_raw))]
print(df_raw[["image_caption"]].head(3))


## 5. Prompt helpers (Phase 1 = baseline notebook style)


In [ ]:
from dataclasses import dataclass
from typing import Optional


SENTIMENT_INSTRUCTION = (
    "Evaluate the sentiment expressed in user reviews and classify each one according to its sentiment rating. "
    "Use a five-point scale: 1-2 negative, 3 neutral, 4-5 positive."
)
RATING_DESCRIPTIONS = {
    1: "Comments show a high level of dissatisfaction and negativity (rating 1).",
    2: "Comments show dissatisfaction (rating 2).",
    3: "Comments are mixed or neutral (rating 3).",
    4: "Comments show satisfaction (rating 4).",
    5: "Comments show strong satisfaction and positivity (rating 5).",
}


@dataclass
class DataConfig:
    use_cot: bool = True
    cot_phrase: str = "Let's take it one step at a time."
    use_one_shot: bool = True


data_cfg = DataConfig()


def format_one_shot(review, rating, cfg: DataConfig):
    desc = RATING_DESCRIPTIONS.get(rating, f"Rating {rating}.")
    return f"Review: {review}\nSentiment (1-5): {rating}. {desc}"


def build_prompt_phase1(review: str, cfg: DataConfig, one_shot_example: Optional[str] = None) -> str:
    parts = [SENTIMENT_INSTRUCTION]
    if cfg.use_cot:
        parts.append(cfg.cot_phrase)
    if one_shot_example and cfg.use_one_shot:
        parts.append("\n\nExample:\n" + one_shot_example)
    parts.append("\n\nReview to classify:\n" + review)
    parts.append(
        '\nAfter your reasoning, end with exactly one line starting with "Sentiment (1-5):" '
        "followed by the rating 1–5 and a short justification (paper-style output)."
    )
    parts.append("\nSentiment (1-5):")
    return "\n".join(parts)


def create_one_shot_pool(samples, cfg: DataConfig, pool_size: int = 5):
    by_rating = {r: [] for r in range(1, 6)}
    for s in samples:
        r = s.get("ground_truth_stars") or s.get("rating")
        if r in by_rating:
            by_rating[r].append(s)
    pool = []
    for r in range(1, 6):
        if by_rating[r]:
            pool.append(random.choice(by_rating[r]))
    if not pool and samples:
        pool = [samples[0]]
    return pool[:pool_size]


def extract_rating_from_output(text: str) -> int:
    text = (text or "").strip()
    if not text:
        return 3
    # Decoded output is usually only new tokens after the prompt ends with "Sentiment (1-5):"
    first_line = text.splitlines()[0].strip() if text else ""
    if re.fullmatch(r"[1-5]", first_line):
        return int(first_line)
    m0 = re.match(r"^\s*([1-5])[\.\:\-]", text)
    if m0:
        return int(m0.group(1))
    m0b = re.match(r"^\s*([1-5])\s+", text)
    if m0b:
        return int(m0b.group(1))
    low = text.lower()
    key = "sentiment (1-5)"
    if key in low:
        idx = low.rfind(key)
        tail = text[idx : idx + 500]
        m = re.search(r"[Ss]entiment\s*\(1-5\)\s*[:=]\s*([1-5])", tail)
        if m:
            return int(m.group(1))
        m = re.search(r"[:=]\s*([1-5])\b", tail)
        if m:
            return int(m.group(1))
    m = re.search(r"[Rr]ating\s*[:=]\s*([1-5])\b", text)
    if m:
        return int(m.group(1))
    for line in reversed(text.splitlines()):
        line = line.strip()
        m = re.match(r"^([1-5])\s*[\.\:\)]", line)
        if m:
            return int(m.group(1))
        m = re.match(r"^([1-5])$", line)
        if m:
            return int(m.group(1))
    m2 = re.search(r"\b([1-5])\b", text)
    if m2:
        return int(m2.group(1))
    digits = re.findall(r"[1-5]", text)
    if digits:
        return int(digits[-1])
    return 3


sample_dicts = df_raw.to_dict("records")
one_shot_pool = create_one_shot_pool(sample_dicts, data_cfg, pool_size=POOL_SIZE)


def pick_one_shot_for_row(row: dict, pool) -> str:
    gt = int(row["ground_truth_stars"])
    candidates = [p for p in pool if int(p["ground_truth_stars"]) != gt]
    if not candidates:
        candidates = pool
    ex = random.choice(candidates)
    return format_one_shot(ex["review_text"], int(ex["ground_truth_stars"]), data_cfg)


## 6. Load LLaMA + PEFT


In [ ]:
import zipfile

_extract_root = "/content" if _IN_COLAB else "."
_zip_candidates = []
for z in (ADAPTER_ZIP, "output_final.zip", "/content/output_final.zip", "./output_final.zip"):
    if z and z not in _zip_candidates:
        _zip_candidates.append(z)
_unzipped = False
for zpath in _zip_candidates:
    if zpath and os.path.isfile(zpath):
        with zipfile.ZipFile(zpath) as z:
            z.extractall(_extract_root)
        print("Unzipped:", zpath, "->", _extract_root)
        _unzipped = True
        break
if not _unzipped:
    print("No zip found; tried:", _zip_candidates)
    print("If adapter is already on disk, ensure ADAPTER_PATH exists.")

if not os.path.isdir(ADAPTER_PATH):
    raise FileNotFoundError(
        f"Adapter not found at {ADAPTER_PATH}. On Colab: upload output_final.zip to /content/ or set ADAPTER_PATH to your unzipped folder."
    )

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig
from peft import PeftModel

try:
    import bitsandbytes  # noqa: F401
    use_4bit = True
except Exception:
    use_4bit = False

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {"trust_remote_code": True, "device_map": "auto", "torch_dtype": torch.float16}
if use_4bit:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()
print("Model ready (4-bit)" if use_4bit else "Model ready (fp16)")


_gen_device = getattr(model, "device", None) or next(model.parameters()).device

# Greedy decode: keep global config consistent (per-call GenerationConfig below avoids stray sampling kwargs).
if getattr(model, "generation_config", None) is not None:
    model.generation_config.do_sample = False


@torch.inference_mode()
def generate_text(prompt: str, max_new_tokens: int, *, truncation_side: str = "right") -> str:
    tok = tokenizer
    prev = tok.truncation_side
    tok.truncation_side = truncation_side
    try:
        inputs = tok(
            prompt, return_tensors="pt", truncation=True, max_length=PROMPT_MAX_LENGTH
        ).to(_gen_device)
    finally:
        tok.truncation_side = prev
    gen_cfg = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
    )
    out = model.generate(**inputs, generation_config=gen_cfg)
    return tok.decode(out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)


## 7. LangGraph: Analyst → RAG Prover → Visual Verifier → Critic

- **RAG Prover:** formats metadata as a fixed "retrieved spec" string (**Gf**).
- **Visual Verifier:** passes BLIP caption as **Ev** (no extra LLM).
- **Critic:** one reconciliation call with the same LoRA model.


In [ ]:
from typing import TypedDict

from langgraph.graph import END, StateGraph

try:
    from langgraph.graph import START as _LG_START

    _USE_START_NODE = True
except ImportError:
    _LG_START = None
    _USE_START_NODE = False


class AgentState(TypedDict, total=False):
    row: dict
    one_shot: str
    analyst_output: str
    analyst_rating: int
    spec_grounding: str
    visual_evidence: str
    critic_output: str
    final_rating: int


def node_analyst(state: AgentState) -> AgentState:
    row = state["row"]
    prompt = build_prompt_phase1(row["review_text"], data_cfg, one_shot_example=state["one_shot"])
    out = generate_text(prompt, MAX_NEW_TOKENS_ANALYST)
    return {
        "analyst_output": out,
        "analyst_rating": extract_rating_from_output(out),
    }


def node_rag_prover(state: AgentState) -> AgentState:
    row = state["row"]
    gf = (
        "Retrieved product specification grounding (Gf) — simulate LanceDB / catalog hit:\n"
        f"- Category: {row.get('main_category', '')}\n"
        f"- Title: {str(row.get('meta_title', ''))[:300]}\n"
        f"- Details: {str(row.get('details', ''))[:500]}"
    )
    return {"spec_grounding": gf}


def node_visual_verifier(state: AgentState) -> AgentState:
    row = state["row"]
    cap = (row.get("image_caption") or "").strip() or "(no image caption — treat visual evidence as missing)"
    ev = f"Visual evidence (Ev) — BLIP image caption (hardware auditor stand-in):\n{cap[:600]}"
    return {"visual_evidence": ev}


def build_critic_prompt(state: AgentState) -> str:
    row = state["row"]
    ao = (state.get("analyst_output") or "")[:600]
    gf = (state.get("spec_grounding") or "")[:700]
    ev = (state.get("visual_evidence") or "")[:700]
    review = (row.get("review_text") or "")[:1800]
    preamble = (
        "Critic pass (same task as training): reconcile the review with product grounding (Gf) and image caption (Ev). "
        "If identity conflicts are clear, adjust the star rating; if Gf/Ev are weak, lean on the Analyst draft.\n\n"
        f"{ev}\n\n{gf}\n\n"
        "Analyst draft (text-only on the same review):\n"
        f"{ao}\n"
    )
    tail = (
        "\n\nReview to classify:\n"
        + review
        + '\nAfter your reasoning, end with exactly one line starting with "Sentiment (1-5):" '
        + "followed by the rating 1–5 and a short justification (paper-style output)."
        + "\nSentiment (1-5):"
    )
    return preamble + tail


def node_critic(state: AgentState) -> AgentState:
    prompt = build_critic_prompt(state)
    out = generate_text(prompt, MAX_NEW_TOKENS_CRITIC, truncation_side="left")
    return {"critic_output": out, "final_rating": extract_rating_from_output(out)}


def build_graph():
    g = StateGraph(AgentState)
    g.add_node("analyst", node_analyst)
    g.add_node("rag_prover", node_rag_prover)
    g.add_node("visual_verifier", node_visual_verifier)
    g.add_node("critic", node_critic)
    if _USE_START_NODE:
        g.add_edge(_LG_START, "analyst")
    else:
        g.set_entry_point("analyst")
    g.add_edge("analyst", "rag_prover")
    g.add_edge("rag_prover", "visual_verifier")
    g.add_edge("visual_verifier", "critic")
    g.add_edge("critic", END)
    return g.compile()


agent_app = build_graph()
print("LangGraph compiled: analyst → rag_prover → visual_verifier → critic")


## 8. Run Phase 1 vs Phase 2 on the same rows

Phase 1: **review text only**. Phase 2 graph: **review + metadata (Gf) + BLIP caption (Ev)**. Checkpoints to **`RESULTS_CSV`** every `CHECKPOINT_EVERY` rows; re-run this cell to **resume** (skips rows already in the CSV by `sample_idx`).


In [ ]:
import os
from tqdm.auto import tqdm

done_idx = set()
results = []
if os.path.isfile(RESULTS_CSV):
    prev = pd.read_csv(RESULTS_CSV)
    if "sample_idx" in prev.columns:
        prev = prev.drop_duplicates(subset=["sample_idx"], keep="last")
        done_idx = set(int(x) for x in prev["sample_idx"].tolist())
        results = prev.to_dict("records")
        print(f"Resume: loaded {len(results)} rows from {RESULTS_CSV}")


def _acc(pred, gt):
    return (pred == gt).mean() if len(gt) else 0.0


def _mae(pred, gt):
    return (pred - gt).abs().mean() if len(gt) else 0.0


n_total = len(df_raw)
for pos in tqdm(range(n_total), desc="LLM eval"):
    if pos in done_idx:
        continue
    row = df_raw.iloc[pos]
    r = row.to_dict()
    one = pick_one_shot_for_row(r, one_shot_pool)

    p1_prompt = build_prompt_phase1(r["review_text"], data_cfg, one_shot_example=one)
    p1_out = generate_text(p1_prompt, MAX_NEW_TOKENS_ANALYST)
    phase1_rating = extract_rating_from_output(p1_out)

    s2 = agent_app.invoke({"row": r, "one_shot": one})
    phase2_rating = int(s2["final_rating"])

    results.append({
        "sample_idx": pos,
        "parent_asin": r.get("parent_asin", ""),
        "gt": int(r["ground_truth_stars"]),
        "phase1": phase1_rating,
        "phase2": phase2_rating,
        "analyst": int(s2["analyst_rating"]),
        "snippet": (r["review_text"] or "")[:70].replace("\n", " "),
    })
    done_idx.add(pos)

    if len(results) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).sort_values("sample_idx").to_csv(RESULTS_CSV, index=False)

pd.DataFrame(results).sort_values("sample_idx").to_csv(RESULTS_CSV, index=False)
df_res = pd.DataFrame(results).sort_values("sample_idx").reset_index(drop=True)

print("\nFirst 12 rows:")
print(df_res.head(12).to_string(index=False))
print("..." if len(df_res) > 12 else "")

a1, a2 = _acc(df_res["phase1"], df_res["gt"]), _acc(df_res["phase2"], df_res["gt"])
m1, m2 = float(_mae(df_res["phase1"], df_res["gt"])), float(_mae(df_res["phase2"], df_res["gt"]))
print(f"\nN = {len(df_res)} | saved → {RESULTS_CSV}")
print(f"Accuracy Phase 1 (review only):     {a1:.4f}")
print(f"Accuracy Phase 2 (review+meta+img): {a2:.4f}")
print(f"MAE Phase 1: {m1:.4f} | MAE Phase 2: {m2:.4f}")
print("\nNote: Phase 2 is not guaranteed to beat Phase 1 on every slice.")


## 9. Plot


In [ ]:
import matplotlib.pyplot as plt

p1_acc = (df_res["phase1"] == df_res["gt"]).mean()
p2_acc = (df_res["phase2"] == df_res["gt"]).mean()
p1_mae = (df_res["phase1"] - df_res["gt"]).abs().mean()
p2_mae = (df_res["phase2"] - df_res["gt"]).abs().mean()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(
    ["Phase 1\n(review only)", "Phase 2\n(review+meta+img)"],
    [p1_acc, p2_acc],
    color=["#4C72B0", "#55A868"],
)
axes[0].set_ylabel("Exact-match accuracy")
axes[0].set_ylim(0, 1)
axes[1].bar(
    ["Phase 1", "Phase 2"],
    [p1_mae, p2_mae],
    color=["#4C72B0", "#55A868"],
)
axes[1].set_ylabel("MAE (stars)")
fig.suptitle("Amazon Reviews 2023 — Phase 1 vs agentic Phase 2")
plt.tight_layout()
plt.show()

m = min(60, len(df_res))
if m > 0:
    sub = df_res.iloc[:m].reset_index(drop=True)
    x = range(m)
    plt.figure(figsize=(12, 3.5))
    plt.plot(x, sub["gt"], "ko-", markersize=4, label="Ground truth")
    plt.plot(x, sub["phase1"], "s-", markersize=4, label="Phase 1")
    plt.plot(x, sub["phase2"], "^-", markersize=4, label="Phase 2")
    plt.ylabel("Stars (1–5)")
    plt.xlabel(f"Sample (first {m} of {len(df_res)})")
    plt.legend()
    plt.title("Per-sample ratings (prefix of eval set)")
    plt.tight_layout()
    plt.show()


## 10. What is / isn’t implemented

| Piece | Status |
|--------|--------|
| LangGraph orchestration | Yes |
| Analyst + Critic (2 LLM calls in graph + 1 for Phase 1 compare) | Yes — Phase 1 uses 1 call; Phase 2 graph uses Analyst + Critic (RAG/Vis nodes are non-LLM) |
| LanceDB + CLIP index | No — metadata string = **toy Gf** |
| LLaVA spatial verifier | No — **BLIP caption = toy Ev** |
| Reflection loop | No — single critic pass (add a conditional edge later) |
